## 05 Tool Calling with MCP
MCP (or Model Context Protocol) creates a open standard protocol between tools and other applications. The basic operation is similar to the standard tool flow in LangChain. The model is provided with the tool description as before, but now it is done via signaling with the MCP server to get the descriptions. And, rather than executing the tool in the tool node, execution takes place on the MCP server when requested by the agent.

Here is a diagram that shows you how:

<div align="center">
<img src="images/05_tools_with_mcp.png" width="450" heigh="500" alt="Tools with MCP"/>
</div>

MCP provides a standardized way to connect AI Agents with external tools and data sources. We can use LangChain MCP Adapters to help us connect to MCP servers for LangChain Agents.

In [1]:
from dotenv import load_dotenv
from rich.console import Console
from rich.markdown import Markdown
from typing import Literal, Union

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool

load_dotenv(override=True)
console = Console()

In [4]:
# lets define our MCP tool connector
# Please pip install langchain-mcp-adapters to use this tool
from langchain_mcp_adapters.client import MultiServerMCPClient
import nest_asyncio
import subprocess
import sys

nest_asyncio.apply()

# Workaround for Windows + Jupyter:
#
# mcp's stdio_client captures sys.stderr as the default `errlog` at *import time*,
# before Jupyter replaces it with its own OutStream. When MCP spawns a subprocess it
# passes that OutStream as `stderr=errlog` to subprocess.Popen, which calls
# errlog.fileno() — but Jupyter's OutStream has no real file descriptor and raises
# UnsupportedOperation: fileno.
#
# Fix: monkey-patch the internal Windows fallback process creator so that any
# errlog that lacks a working fileno() is silently replaced with subprocess.PIPE.
if sys.platform == "win32":
    import mcp.os.win32.utilities as _mcp_win32

    _orig_fallback = _mcp_win32._create_windows_fallback_process

    async def _patched_fallback(command, args, env, errlog, cwd):
        try:
            if errlog is not None:
                errlog.fileno()
        except Exception:
            errlog = subprocess.PIPE
        return await _orig_fallback(command, args, env, errlog, cwd)

    _mcp_win32._create_windows_fallback_process = _patched_fallback

mcp_client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "npx",
            "args": ["-y", "@theo.foobar/mcp-time"],
        }
    },
)
mcp_tools = await mcp_client.get_tools()
print(f"Loaded {len(mcp_tools)} -> {[tool.name for tool in mcp_tools]}")

  + Exception Group Traceback (most recent call last):
  |   File "c:\Users\BHOBEMRMANISHJAGDISH\Dev\code\git_projects\learning_langchain\.venv\Lib\site-packages\IPython\core\interactiveshell.py", line 3745, in run_code
  |     await eval(code_obj, self.user_global_ns, self.user_ns)
  |   File "C:\Users\BHOBEMRMANISHJAGDISH\AppData\Local\Temp\ipykernel_17212\1046017776.py", line 44, in <module>
  |     mcp_tools = await mcp_client.get_tools()
  |                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "c:\Users\BHOBEMRMANISHJAGDISH\Dev\code\git_projects\learning_langchain\.venv\Lib\site-packages\langchain_mcp_adapters\client.py", line 197, in get_tools
  |     tools_list = await asyncio.gather(*load_mcp_tool_tasks)
  |                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "C:\Users\BHOBEMRMANISHJAGDISH\AppData\Roaming\uv\python\cpython-3.12.11-windows-x86_64-none\Lib\asyncio\tasks.py", line 385, in __wakeup
  |     future.result()
  |   File "C:\Users\BHOBEMRMANISHJAGD